In [4]:
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from lightgbm import LGBMClassifier
from scipy.stats import randint, uniform
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import os
import joblib

# ============================================================
# 3. LightGBM Halving Random Search (MUCH FASTER)
# ============================================================
param_dist = {
    "n_estimators": randint(100, 400),  # capped lower
    "learning_rate": uniform(0.01, 0.15),
    "max_depth": randint(3, 6),
    "num_leaves": randint(16, 48),
    "subsample": uniform(0.8, 0.2),
    "colsample_bytree": uniform(0.8, 0.2),
    "reg_alpha": uniform(0.0, 0.1),
    "reg_lambda": uniform(0.0, 0.1)
}

lgbm = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Successive halving will try fewer candidates first, then focus on the best
halving_search = HalvingRandomSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    factor=2,               # how aggressively to prune configs
    resource='n_estimators', # early stopping by n_estimators
    max_resources=400,      # cap trees
    min_resources=50,       # quick first trials
    cv=2,                   # lighter CV
    scoring="roc_auc",
    random_state=42,
    verbose=2,
    n_jobs=-1
)

print("\n🔎 Running HalvingRandomSearchCV for LightGBM...")
halving_search.fit(X_train, y_train)

print(f"\n✅ Best Parameters: {halving_search.best_params_}")
print(f"✅ Best CV Score (AUC): {halving_search.best_score_:.4f}")

# ============================================================
# 4. Evaluate Best Model
# ============================================================
best_model = halving_search.best_estimator_

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

test_acc = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_proba)

print("\n📊 Final Evaluation on Test Set:")
print(f"Accuracy: {test_acc:.4f}")
print(f"AUC: {test_auc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# ============================================================
# 5. Save Best Model
# ============================================================
# Define MODEL_DIR if it's not already defined
MODEL_DIR = os.path.join(os.getcwd(), 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(
    MODEL_DIR,
    f"LightGBM_TUNED_ACC{test_acc:.4f}_AUC{test_auc:.4f}.pkl"
)
joblib.dump(best_model, model_path)

print(f"\n💾 Tuned LightGBM model saved at: {model_path}")



🔎 Running HalvingRandomSearchCV for LightGBM...
n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 4
min_resources_: 50
max_resources_: 400
aggressive_elimination: False
factor: 2
----------
iter: 0
n_candidates: 8
n_resources: 50
Fitting 2 folds for each of 8 candidates, totalling 16 fits
----------
iter: 1
n_candidates: 4
n_resources: 100
Fitting 2 folds for each of 4 candidates, totalling 8 fits
----------
iter: 2
n_candidates: 2
n_resources: 200
Fitting 2 folds for each of 2 candidates, totalling 4 fits
----------
iter: 3
n_candidates: 1
n_resources: 400
Fitting 2 folds for each of 1 candidates, totalling 2 fits

✅ Best Parameters: {'colsample_bytree': np.float64(0.8749080237694725), 'learning_rate': np.float64(0.15260714596148742), 'max_depth': 5, 'num_leaves': 23, 'reg_alpha': np.float64(0.05986584841970366), 'reg_lambda': np.float64(0.015601864044243652), 'subsample': np.float64(0.8311989040672406), 'n_estimators': 400}
✅ Best CV Score (AUC): 0.9926

📊 Final Evalua

In [5]:
# ============================================================
# 🔍 Data Leakage Check
# ============================================================

import pandas as pd
import numpy as np

def check_data_leakage(df, target_col="Label", threshold=0.85):
    """
    Detects potential data leakage by checking correlation between features and target.
    Args:
        df (pd.DataFrame): Dataset including target column
        target_col (str): Name of the target column
        threshold (float): Correlation above this value will trigger a warning
    """
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe")

    # Encode categorical features temporarily
    temp_df = df.copy()
    for col in temp_df.select_dtypes(include=["object"]).columns:
        temp_df[col] = temp_df[col].astype("category").cat.codes

    # Compute correlations with target
    corr = temp_df.corr(numeric_only=True)[target_col].drop(target_col)

    # Sort by absolute correlation
    corr_sorted = corr.abs().sort_values(ascending=False)

    print("\n🔎 Potential Data Leakage Check:")
    suspicious = []
    for feature, value in corr_sorted.items():
        if value > threshold:
            suspicious.append((feature, value))
            print(f"⚠️  Feature '{feature}' has HIGH correlation with target ({value:.4f})")

    if not suspicious:
        print("✅ No strong evidence of data leakage found.")
    else:
        print("\n⚠️ WARNING: Some features may leak label information! Consider removing them.")

    return suspicious


# ============================================================
# Run leakage check on your dataset
# ============================================================
suspect_features = check_data_leakage(df, target_col="Label", threshold=0.85)



🔎 Potential Data Leakage Check:
✅ No strong evidence of data leakage found.


In [6]:
# ============================================================
# 🔍 Advanced Data Leakage Check (Correlation + MI)
# ============================================================

import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

def check_data_leakage(df, target_col="Label", corr_threshold=0.85, mi_threshold=0.5):
    """
    Detect potential data leakage by using:
    1. Pearson correlation (linear relationships)
    2. Mutual Information (non-linear relationships)
    """

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe")

    temp_df = df.copy()
    y = temp_df[target_col]

    # Encode categorical features
    for col in temp_df.select_dtypes(include=["object"]).columns:
        temp_df[col] = temp_df[col].astype("category").cat.codes

    X = temp_df.drop(columns=[target_col])

    # --- Step 1: Correlation Test ---
    corr = temp_df.corr(numeric_only=True)[target_col].drop(target_col)
    corr_sorted = corr.abs().sort_values(ascending=False)

    print("\n📊 Correlation Check:")
    suspicious_corr = []
    for feature, value in corr_sorted.items():
        if value > corr_threshold:
            suspicious_corr.append((feature, value))
            print(f"⚠️  Feature '{feature}' HIGH correlation with target ({value:.4f})")

    if not suspicious_corr:
        print("✅ No suspiciously high correlations found.")

    # --- Step 2: Mutual Information Test ---
    print("\n📊 Mutual Information Check:")
    mi = mutual_info_classif(X, y, discrete_features="auto", random_state=42)
    mi_scores = pd.Series(mi, index=X.columns).sort_values(ascending=False)

    suspicious_mi = mi_scores[mi_scores > mi_threshold]
    if not suspicious_mi.empty:
        for feature, score in suspicious_mi.items():
            print(f"⚠️  Feature '{feature}' HIGH MI score ({score:.4f}) — possible leakage")
    else:
        print("✅ No suspicious MI scores detected.")

    return suspicious_corr, suspicious_mi


# ============================================================
# Run leakage check
# ============================================================
susp_corr, susp_mi = check_data_leakage(df, target_col="Label", corr_threshold=0.85, mi_threshold=0.5)



📊 Correlation Check:
✅ No suspiciously high correlations found.

📊 Mutual Information Check:
⚠️  Feature 'Protein_ID' HIGH MI score (0.5689) — possible leakage


In [14]:
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from lightgbm import LGBMClassifier
from scipy.stats import randint, uniform
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import os
import joblib

# ============================================================
# 0. Drop Leakage Column (Protein_ID)
# ============================================================
if "Protein_ID" in X_train.columns:
    print("⚠️ Dropping Protein_ID to prevent data leakage...")
    X_train = X_train.drop(columns=["Protein_ID"])
    X_test = X_test.drop(columns=["Protein_ID"])

# ============================================================
# 1. LightGBM Halving Random Search (FAST & SAFE)
# ============================================================
param_dist = {
    "learning_rate": uniform(0.01, 0.15),
    "max_depth": randint(3, 6),
    "num_leaves": randint(16, 48),
    "subsample": uniform(0.8, 0.2),
    "colsample_bytree": uniform(0.8, 0.2),
    "reg_alpha": uniform(0.0, 0.1),
    "reg_lambda": uniform(0.0, 0.1)
}

lgbm = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

halving_search = HalvingRandomSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    factor=2,                # how aggressively to prune configs
    resource="n_estimators", # halving will increase #trees
    max_resources=400,       # cap trees
    min_resources=50,        # quick early tests
    cv=2,                    # lighter CV for speed
    scoring="roc_auc",
    random_state=42,
    verbose=2,
    n_jobs=-1
)

print("\n🔎 Running HalvingRandomSearchCV for LightGBM...")
halving_search.fit(X_train, y_train)

print(f"\n✅ Best Parameters: {halving_search.best_params_}")
print(f"✅ Best CV Score (AUC): {halving_search.best_score_:.4f}")

# ============================================================
# 2. Evaluate Best Model
# ============================================================
best_model = halving_search.best_estimator_

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

test_acc = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_proba)

print("\n📊 Final Evaluation on Test Set:")
print(f"Accuracy: {test_acc:.4f}")
print(f"AUC: {test_auc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# ============================================================
# 3. Save Best Model
# ============================================================
MODEL_DIR = os.path.join(os.getcwd(), 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(
    MODEL_DIR,
    f"LightGBM_TUNED_noProteinID_ACC{test_acc:.4f}_AUC{test_auc:.4f}.pkl"
)
joblib.dump(best_model, model_path)

print(f"\n💾 Tuned LightGBM model saved at: {model_path}")



🔎 Running HalvingRandomSearchCV for LightGBM...
n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 4
min_resources_: 50
max_resources_: 400
aggressive_elimination: False
factor: 2
----------
iter: 0
n_candidates: 8
n_resources: 50
Fitting 2 folds for each of 8 candidates, totalling 16 fits
----------
iter: 1
n_candidates: 4
n_resources: 100
Fitting 2 folds for each of 4 candidates, totalling 8 fits
----------
iter: 2
n_candidates: 2
n_resources: 200
Fitting 2 folds for each of 2 candidates, totalling 4 fits
----------
iter: 3
n_candidates: 1
n_resources: 400
Fitting 2 folds for each of 1 candidates, totalling 2 fits

✅ Best Parameters: {'colsample_bytree': np.float64(0.8116167224336399), 'learning_rate': np.float64(0.13992642186624027), 'max_depth': 5, 'num_leaves': 37, 'reg_alpha': np.float64(0.005641157902710026), 'reg_lambda': np.float64(0.07219987722668247), 'subsample': np.float64(0.9877105418031501), 'n_estimators': 400}
✅ Best CV Score (AUC): 0.9542

📊 Final Evalua

In [15]:
# ============================================================
# ⚡ Final LightGBM Training with Best Parameters (No Search)
# ============================================================

import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from lightgbm import LGBMClassifier
import joblib
from google.colab import drive

# ============================================================
# 1. Mount Google Drive & Setup Paths
# ============================================================
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/MBP_PREDICTOR"
DATA_PATH = os.path.join(BASE_PATH, "data/features_cleaned_final.csv")
MODEL_DIR = os.path.join(BASE_PATH, "TRAINED_MODELS")
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"📂 Loading dataset from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
print("✅ Dataset shape:", df.shape)

# ============================================================
# 2. Prepare Features & Target
# ============================================================
target_col = "Label"

# Drop Protein_ID if present (avoid leakage)
if "Protein_ID" in df.columns:
    print("⚠️ Dropping Protein_ID column to prevent leakage.")
    df = df.drop(columns=["Protein_ID"])

X = df.drop(columns=[target_col])
y = df[target_col]

# Fix column names for LightGBM
X.columns = [str(c).replace("[", "_").replace("]", "_").replace("<", "_") for c in X.columns]

# Encode categorical features
for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ============================================================
# 3. Train LightGBM with Pre-Selected Best Parameters
# ============================================================
best_params = {
    'colsample_bytree': 0.8116167224336399,
    'learning_rate': 0.13992642186624027,
    'max_depth': 5,
    'num_leaves': 37,
    'reg_alpha': 0.005641157902710026,
    'reg_lambda': 0.07219987722668247,
    'subsample': 0.9877105418031501,
    'n_estimators': 400,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print("\n🚀 Training LightGBM with best parameters (no search)...")
best_model = LGBMClassifier(**best_params)
best_model.fit(X_train, y_train)

# ============================================================
# 4. Evaluate Model
# ============================================================
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

test_acc = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_proba)

print("\n📊 Final Evaluation on Test Set:")
print(f"Accuracy: {test_acc:.4f}")
print(f"AUC: {test_auc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# ============================================================
# 5. Save Model
# ============================================================
model_path = os.path.join(
    MODEL_DIR,
    f"LightGBM_FINAL_noProteinID_ACC{test_acc:.4f}_AUC{test_auc:.4f}.pkl"
)
joblib.dump(best_model, model_path)

print(f"\n💾 Final LightGBM model saved at: {model_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Loading dataset from: /content/drive/MyDrive/MBP_PREDICTOR/data/features_cleaned_final.csv
✅ Dataset shape: (3072, 1057)
⚠️ Dropping Protein_ID column to prevent leakage.

🚀 Training LightGBM with best parameters (no search)...

📊 Final Evaluation on Test Set:
Accuracy: 0.9122
AUC: 0.9773
              precision    recall  f1-score   support

           0       0.94      0.88      0.91       308
           1       0.89      0.94      0.91       307

    accuracy                           0.91       615
   macro avg       0.91      0.91      0.91       615
weighted avg       0.91      0.91      0.91       615


💾 Final LightGBM model saved at: /content/drive/MyDrive/MBP_PREDICTOR/TRAINED_MODELS/LightGBM_FINAL_noProteinID_ACC0.9122_AUC0.9773.pkl
